# Starlink broadband: real-orbit CSV export

Read the local snapshot fixed by `download_tle.ipynb` and propagate it with Skyfield/SGP4. This notebook works offline. Apply the reference code's `[DTC]` exclusion rule and 450 km altitude threshold. Altitude filtering is an engineering approximation; TLE data alone cannot establish operational service status.


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

# Locate the shared module from either the repository or notebook directory.
ROOT = next((p for base in [Path.cwd(), *Path.cwd().parents]
             for p in [base, base / "generate satellite"]
             if (p / "satellite_pipeline.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the WorldNTN repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from satellite_pipeline import (
    download_snapshot, initialize_scenario, read_scenario, load_snapshot,
    export_tracks, preview_export, merge_exports, stamp,
)
# If dependencies are missing, run this in a separate notebook cell:
# %pip install -r "{ROOT / 'requirements.txt'}"
print("Workspace:", ROOT)


In [ ]:
CONSTELLATION = "starlink"
scenario, times = read_scenario(ROOT)
print("UTC range:", times[0], "to", times[-1], "samples:", len(times))
print("Minimum orbit altitude (km):", scenario["min_orbit_altitude_km"][CONSTELLATION])
display(scenario)


## Load the real satellite catalog

Load the complete OMM JSON by default. To read a legacy TLE file explicitly, use `load_tle_file(path, CONSTELLATION)`; traditional TLEs may omit six-digit IDs.


In [ ]:
satellites = load_snapshot(CONSTELLATION, root=ROOT)
print("Catalog satellites:", len(satellites))


## Generate CSV files

Export every visible satellite that meets the thresholds, without top-N truncation. Preserve the original ENU columns and add ECEF position/velocity, latitude/longitude, range rate, unique IDs, time indices, and provenance. Setting `write_all_states=true` in `scenario.json` also saves retained satellite states globally, including satellites invisible to the observer; these files can be large. Velocities come directly from propagation and rotating-frame transformations.


In [ ]:
result = export_tracks(satellites, CONSTELLATION, root=ROOT)
display(preview_export(result))


In [ ]:
counts = pd.read_csv(result["counts_csv"])
audit = pd.read_csv(result["audit_csv"])
display(counts.describe())
display(audit.head(10))
print("Audit:", result["audit_csv"])


Times with zero visible satellites are included in `counts.csv` and in the averages. `filter_audit.csv` records filtering reasons, stale epochs, and propagation error counts; `metadata.json` records the complete configuration and input hashes.
